## Structure Output

Models can be requested to provide their response in a format matching a given schema

## Pydantic



In [1]:
import os
from langchain.chat_models import init_chat_model
os.environ['GROQ_API_KEY']=os.getenv('GROQ_API_KEY')
model=init_chat_model("groq:qwen/qwen3-32b")
model

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001EBAD7C0440>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001EBAD7C1160>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [6]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="The year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The movie rating out of 10")
    

In [7]:
model_with_structure=model.with_structured_output(Movie)
model_with_structure

RunnableBinding(bound=ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001EBAD7C0440>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001EBAD7C1160>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'The year the movie was released', 'type': 'integer'}, 'director': {'description': 'The director of the movie', 'type': 'string'}, 'rating': {'description': 'The movie rating out of 10', 'type': 'number'}}, 'required': ['title', 'year', 'di

In [8]:
model.invoke("Provide the details about the movie Inception")

AIMessage(content='<think>\nOkay, so I need to provide details about the movie Inception. Let me start by recalling what I know about it. Inception is a 2010 movie directed by Christopher Nolan. The main cast includes Leonardo DiCaprio, Joseph Gordon-Levitt, Ellen Page, and others. It\'s a sci-fi action film that involves dreams within dreams. \n\nThe plot revolves around Dom Cobb, played by DiCaprio, who is a professional thief who steals information by entering the subconscious. Instead of stealing, he\'s asked to plant an idea into someone\'s mind, which is called "inception." The concept is pretty complex, involving multiple layers of dreams. There\'s the idea of a totem, which is an object that helps characters know if they\'re in a dream or reality. Cobb\'s totem is a spinning top.\n\nI remember there\'s a lot of action sequences, especially one where Arthur (played by Tom Hardy) does a fight in a zero-gravity environment. Joseph Gordon-Levitt plays Arthur\'s partner, Ariadne, wh

In [9]:
model_with_structure.invoke("Provide the details about the movie Inception")

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

## Message output alongside parsed structure

In [10]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="The year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The movie rating out of 10")
    
model_with_structure=model.with_structured_output(Movie, include_raw=True)

response=model_with_structure.invoke("Provide details of ovie Inception")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for details about the movie Inception. Let me check the tools available. There\'s a Movie function that requires title, year, director, and rating. I need to fill these parameters for Inception. The title is clearly "Inception". The year it was released is 2010. The director is Christopher Nolan. As for the rating, I think it\'s around 8.8 on IMDb. Let me confirm that. Yes, IMDb lists it at 8.8. So I should structure the tool call with these details. Make sure all required fields are included and the JSON is correctly formatted.\n', 'tool_calls': [{'id': 'ey5rhrss5', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 179, 'prompt_tokens': 225, 'total_tokens': 404, 'completion_time': 0.330423847, 'completion_tokens_details': {'reasoning_token

### Nested structure

In [11]:
from pydantic import BaseModel, Field
class Actor(BaseModel):
    name:str
    role:str

class MovieDetails(BaseModel):
    title:str
    year:int
    cast:list[Actor]
    genres:list[str]
    budget:float | None = Field(None, description="Budget in million USD")

    


In [16]:
model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about Bahulbali the beginning")
response

MovieDetails(title='Baahubali: The Beginning', year=2015, cast=[Actor(name='Prabhas', role='Baahubali'), Actor(name='Rana Daggubati', role='Bhallaldeva'), Actor(name='Tamannaah', role='Devasena'), Actor(name='Anushka Shetty', role='Avanthika')], genres=['Action', 'Adventure', 'Fantasy'], budget=100.0)

### TypedDict

In [17]:
from typing_extensions import TypedDict, Annotated


In [18]:
class MovieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]

In [20]:
model_with_typeDict =model.with_structured_output(MovieDict)
response=model_with_typeDict.invoke("please provide information about movie prem ratan dhan payo")
response

{'director': 'Sooraj Barjatya',
 'rating': 6.5,
 'title': 'Prem Ratan Dhan Payo',
 'year': 2015}

In [21]:
class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie prem ratan dhan payo")
response

{'budget': 200,
 'cast': [{'name': 'Salman Khan', 'role': 'Ratan'},
  {'name': 'Sonam Kapoor', 'role': 'Dia'}],
 'genres': ['Romance', 'Drama'],
 'title': 'Prem Ratan Dhan Payo',
 'year': 2015}

### Dataclasses

In [22]:

from pydantic import BaseModel, Field
from langchain.agents import create_agent

In [23]:
class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model="groq:qwen/qwen3-32b",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})


In [24]:
result

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='6a770a03-19ea-4375-8c1d-16fdb1f6cb93'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, let me see. The user wants me to extract contact information from the given text: "John Doe, john@example.com, (555) 123-4567". The tools provided include a ContactInfo function with parameters for name, email, and phone. All three are required.\n\nFirst, I need to parse the input. The name is "John Doe", which is straightforward. The email is "john@example.com". The phone number is "(555) 123-4567". I should check if the phone number is in the correct format. The function\'s parameters don\'t specify a particular format, so as long as it\'s a string, it should be acceptable. \n\nI need to make sure all required fields are present. The input has all three: name, email, phone. So I can call the ContactInfo function with thes